# 🇧🇷 OpenMythos para Iniciantes
## RDT (Recurrent-Depth Transformer) - Tutorial Completo

Este notebook vai te ensinar a:
1. Carregar a arquitetura OpenMythos
2. Criar um dataset minúsculo (100 frases)
3. Treinar por 1 época (só para ver o loss diminuir)
4. Gerar texto com o modelo treinado

**Nenhuma instalação local necessária!** Tudo roda no seu navegador.

---

## ⚙️ Passo 1: Instalar o OpenMythos

In [ ]:
!pip install open-mythos -q
print("✅ OpenMythos instalado!")

## 📦 Passo 2: Criar configuração reduzida para treino rápido

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

# Importar o OpenMythos
from open_mythos import MythosConfig, OpenMythos

# Configuração MINÚSCULA para rodar rápido no Colab (GPU grátis)
# Menor que 1M parâmetros — treina em segundos
cfg = MythosConfig(
    vocab_size=1000,          # vocabulário pequeno
    dim=128,                  # hidden dimension reduzido
    n_heads=4,                # número de cabeças de atenção
    n_kv_heads=2,             # cabeças KV (GQA)
    max_seq_len=64,           # sequências curtas
    max_loop_iters=3,         # RDT: 3 loops de raciocínio
    prelude_layers=1,         # 1 camada antes do loop
    coda_layers=1,            # 1 camada depois do loop
    attn_type="gqa",          # GQA é mais leve que MLA
    n_experts=4,              # MoE com 4 experts
    n_shared_experts=1,       # 1 expert compartilhado
    n_experts_per_tok=2,      # top-2 por token
    expert_dim=64,            # dimensão interna dos experts
    lora_rank=4,              # LoRA de profundidade
    act_threshold=0.99,       # halting threshold
)

print("🔧 Configuração:")
print(f"   - Dimensão: {cfg.dim}")
print(f"   - Head: {cfg.n_heads}")
print(f"   - Loops: {cfg.max_loop_iters}")
print(f"   - Experts: {cfg.n_experts}")
print(f"   - Atenção: {cfg.attn_type.upper()}")

# Criar o modelo
model = OpenMythos(cfg)

# Contar parâmetros
total_params = sum(p.numel() for p in model.parameters())
print(f"\n✅ Modelo criado! Parâmetros: {total_params:,}")

# Mover para GPU se disponível
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(f"📱 Rodando em: {device}")

## 📝 Passo 3: Criar dataset minúsculo (100 frases em português)

In [ ]:
# Frases simples em português
frases = [
    "o ceu e azul",
    "o sol brilha forte",
    "a lua esta cheia",
    "as estrelas brilham",
    "o dia e claro",
    "a noite e escura",
    "o passaro voa",
    "o peixe nada",
    "o gato mia",
    "o cachorro late",
    "a casa e grande",
    "o carro e vermelho",
    "a flor e bonita",
    "a arvore e alta",
    "a agua e limpa",
    "o fogo queima",
    "o vento sopra",
    "a chuva cai",
    "o rio corre",
    "o mar e salgado",
]

# Duplicar para ter 100 exemplos
frases = frases * 5  # 20 x 5 = 100 frases

print(f"📚 Dataset criado com {len(frases)} frases")
print("\n📖 Exemplos:")
for i in range(5):
    print(f"   {i+1}. {frases[i]}")

## 🔤 Passo 4: Criar tokenizador simples

In [ ]:
# Tokenizador manual (sem instalar nada extra)
class SimpleTokenizer:
    def __init__(self, textos, vocab_size=1000):
        # Tokens especiais nos primeiros índices
        self.vocab = {"<PAD>": 0, "<UNK>": 1, "<BOS>": 2, "<EOS>": 3}
        
        # Extrair palavras únicas
        palavras = set()
        for texto in textos:
            for palavra in texto.split():
                palavras.add(palavra)
        
        # Mapear palavras para índices (começando depois dos especiais)
        for palavra in sorted(palavras):
            idx = len(self.vocab)
            if idx >= vocab_size:
                break
            self.vocab[palavra] = idx
        
        self.inv_vocab = {v: k for k, v in self.vocab.items()}
    
    def encode(self, texto):
        tokens = [self.vocab["<BOS>"]]
        for palavra in texto.split():
            tokens.append(self.vocab.get(palavra, self.vocab["<UNK>"]))
        tokens.append(self.vocab["<EOS>"])
        return tokens
    
    def decode(self, tokens):
        palavras = []
        for t in tokens:
            if isinstance(t, torch.Tensor):
                t = t.item()
            p = self.inv_vocab.get(t, "<UNK>")
            if p not in ("<PAD>", "<BOS>", "<EOS>", "<UNK>"):
                palavras.append(p)
        return " ".join(palavras)

tokenizer = SimpleTokenizer(frases, vocab_size=cfg.vocab_size)
print(f"📖 Vocabulário: {len(tokenizer.vocab)} tokens (de {cfg.vocab_size} disponíveis)")
print(f"   Palavras: {list(tokenizer.inv_vocab.values())}")

## 🎯 Passo 5: Dataset PyTorch

In [ ]:
class FrasesDataset(Dataset):
    def __init__(self, frases, tokenizer, max_len=32):
        self.data = []
        for frase in frases:
            tokens = tokenizer.encode(frase)
            # Truncar se necessário
            if len(tokens) > max_len:
                tokens = tokens[:max_len]
            # Padding até max_len
            pad_len = max_len - len(tokens)
            tokens = tokens + [0] * pad_len  # <PAD> = 0
            
            # Input = tokens[:-1], target = tokens[1:]
            self.data.append({
                "input": torch.tensor(tokens[:-1], dtype=torch.long),
                "target": torch.tensor(tokens[1:], dtype=torch.long)
            })
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]["input"], self.data[idx]["target"]

# Criar dataset e dataloader
dataset = FrasesDataset(frases, tokenizer, max_len=cfg.max_seq_len)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

print(f"✅ Dataset pronto: {len(dataset)} exemplos")
print(f"   Batch size: 8 → {len(dataloader)} batches/época")

# Ver um exemplo
inputs, targets = next(iter(dataloader))
print(f"\n📊 Exemplo de batch:")
print(f"   Input shape: {inputs.shape}")
print(f"   Target shape: {targets.shape}")
print(f"   Input decodificado: '{tokenizer.decode(inputs[0])}'")

## 🚀 Passo 6: Treinar por 1 época

In [ ]:
# Otimizador
optimizer = optim.AdamW(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

print("🎯 Iniciando treino (", len(dataloader), " batches)..." )
print("="*50)

model.train()
total_loss = 0.0

for batch_idx, (inputs, targets) in enumerate(dataloader):
    inputs = inputs.to(device)
    targets = targets.to(device)
    
    optimizer.zero_grad()
    
    # Forward com RDT: n_loops = max_loop_iters
    outputs = model(inputs, n_loops=cfg.max_loop_iters)
    
    # outputs: (B, T, vocab_size), targets: (B, T)
    loss = criterion(outputs.reshape(-1, cfg.vocab_size), targets.reshape(-1))
    
    loss.backward()
    optimizer.step()
    
    total_loss += loss.item()
    
    if (batch_idx + 1) % max(1, len(dataloader)//5) == 0 or batch_idx == 0:
        print(f"   Batch {batch_idx+1:2d}/{len(dataloader)} | Loss: {loss.item():.4f}")

avg_loss = total_loss / len(dataloader)
print("="*50)
print(f"✅ Treino concluído! Loss médio: {avg_loss:.4f}")
if avg_loss < 2.0:
    print("   🎉 O modelo está aprendendo!")
else:
    print("   ⚠️ Loss ainda alto — aumente dados ou épocas")

## 💬 Passo 7: Gerar texto com o modelo treinado

In [ ]:
def gerar_texto(prompt, max_tokens=15):
    model.eval()
    with torch.no_grad():
        tokens = tokenizer.encode(prompt)
        input_ids = torch.tensor([tokens], device=device)
        
        # Usar o método generate do OpenMythos com temperature=0.8
        output_ids = model.generate(
            input_ids,
            max_new_tokens=max_tokens,
            n_loops=cfg.max_loop_iters,
            temperature=0.8,
            top_k=30
        )
        
        return tokenizer.decode(output_ids[0].tolist())

print("🔮 Gerando textos após treino:")
print("-"*50)

testes = ["o ceu", "o gato", "a agua"]
for prompt in testes:
    resultado = gerar_texto(prompt, max_tokens=12)
    print(f"\n📝 Prompt: '{prompt}'")
    print(f"   Gerado: '{resultado}'")

## 📈 Passo 8: Análise — O que aprendemos?

In [ ]:
print("="*60)
print("📊 RESUMO DO EXPERIMENTO")
print("="*60)
print(f"""
✅ O que você aprendeu:

1. ARQUITETURA RDT (Recurrent-Depth Transformer)
   • Prelude ({cfg.prelude_layers} camada) → Recurrent Block ({cfg.max_loop_iters} loops) → Coda ({cfg.coda_layers} camada)
   • O modelo "pensa" {cfg.max_loop_iters}x por token sem gerar texto intermediário
   • Atenção: {cfg.attn_type.upper()}
   • MoE: {cfg.n_experts} experts, top-{cfg.n_experts_per_tok} por token

2. TREINO REAL (1 época)
   • Dataset: {len(frases)} frases em português
   • Loss final: {avg_loss:.4f}
   • {'✅ Aprendendo!' if avg_loss < 2 else '⚠️ Precisa de mais tempo'}

3. ESTABILIDADE LTI
   • A matriz A tem raio espectral < 1 por construção
   • Isso evita explosão do gradiente nos loops

4. PRÓXIMOS PASSOS
   • Aumente o dataset (1000+ frases)
   • Rode 10-50 épocas
   • Teste attn_type='mla'
   • Salve: torch.save(model.state_dict(), 'modelo.pt')
""")